# Practical: Measuring the Cost of the KV Cache in a Hugging Face Transformer

**Course:** LLM Applications  
**Topic:** KV caching, long-context inference, and autoregressive decoding

## Learning objectives

By the end of this practical you should be able to:

1. Explain the difference between **prefill** and **decode**.
2. Measure how prefill latency changes with input context length.
3. Measure how decode latency changes as the KV cache grows.
4. Estimate the memory occupied by the KV cache.
5. Relate observed measurements to the theoretical scaling laws.
6. Identify whether a model uses **Multi-Head Attention (MHA)**, **Grouped-Query Attention (GQA)**, or **Multi-Query Attention (MQA)**.
7. Explain why real hardware measurements are not perfectly linear.

> **Important:** This notebook is designed to run on a laptop CPU, Apple Silicon, NVIDIA GPU, or Google Colab. The exact timings will depend strongly on hardware, software versions, and background load. We are interested primarily in the *trend*.

## 1. The central question

Suppose an LLM has already generated a long conversation.

When the next token is generated:

- Why doesn't the model recompute all previous keys and values?
- Why does the KV cache grow with context length?
- Why can each new token become more expensive as the conversation grows?

The simplified picture is:

\[
\text{KV cache memory} \propto T
\]

where \(T\) is the number of cached tokens.

During decode, the new query must attend over the cached keys and values:

\[
Q_t K_{1:t}^{T}
\]

so the amount of attention work for each newly generated token grows approximately with \(T\).

However, real systems are more complicated. GPU kernels, memory bandwidth, cache effects, batching, and implementation details mean that measured latency will **not** necessarily be a perfect straight line.

Our goal is to measure what actually happens.

In [ ]:
# If needed in Colab or a fresh environment, uncomment:
# %pip install -q transformers accelerate psutil pandas matplotlib

import time
import gc
import math
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Platform:", platform.platform())

## 2. Choose a small causal language model

We use `Qwen/Qwen2.5-0.5B-Instruct` as a compact example.

It is small enough for teaching and experimentation, while still exposing useful attention configuration information.

If downloading this model is inconvenient, you can replace `MODEL_ID` with another small causal LM available through Hugging Face.

For a CPU-only laptop, you may prefer a very small model. For Colab with a GPU, you can experiment with a somewhat larger model after completing the basic exercise.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
)

model = model.to(device)
model.eval()

print("Loaded:", MODEL_ID)
print("Device:", device)

## 3. Inspect the model's attention architecture

The most important configuration fields are:

- `num_attention_heads`: number of query attention heads
- `num_key_value_heads`: number of key/value heads

If:

\[
H_{KV} = H_Q
\]

the model uses ordinary **Multi-Head Attention (MHA)**.

If:

\[
1 < H_{KV} < H_Q
\]

the model uses **Grouped-Query Attention (GQA)**.

If:

\[
H_{KV} = 1
\]

the model uses **Multi-Query Attention (MQA)**.

This matters because the KV cache stores K and V for the **KV heads**, not independently for every query head.

In [ ]:
config = model.config

n_layers = getattr(config, "num_hidden_layers", None)
n_q_heads = getattr(config, "num_attention_heads", None)
n_kv_heads = getattr(config, "num_key_value_heads", n_q_heads)
head_dim = getattr(config, "head_dim", None)

if head_dim is None and n_q_heads is not None:
    hidden_size = getattr(config, "hidden_size", None)
    head_dim = hidden_size // n_q_heads

if n_kv_heads == n_q_heads:
    attention_type = "MHA"
elif n_kv_heads == 1:
    attention_type = "MQA"
else:
    attention_type = "GQA"

print("Attention architecture:", attention_type)
print("Transformer layers:", n_layers)
print("Query heads:", n_q_heads)
print("KV heads:", n_kv_heads)
print("Head dimension:", head_dim)

## 4. Estimating KV-cache memory

For a standard decoder Transformer, the cache contains K and V tensors approximately shaped like:

\[
[\text{batch}, H_{KV}, T, d]
\]

for each layer.

The approximate number of stored elements is therefore:

\[
2 \times L \times B \times H_{KV} \times T \times d
\]

where:

- \(L\) = number of layers
- \(B\) = batch size
- \(H_{KV}\) = number of KV heads
- \(T\) = sequence length
- \(d\) = head dimension

If each element uses \(b\) bytes:

\[
M_{KV}
=
2L B H_{KV} T d b
\]

This is a **theoretical estimate**. Actual allocated GPU memory can be higher because of allocator overhead, temporary tensors, fragmentation, and implementation-specific cache layouts.

In [ ]:
def theoretical_kv_cache_bytes(
    layers,
    kv_heads,
    sequence_length,
    head_dim,
    batch_size=1,
    bytes_per_element=2,
):
    return (
        2
        * layers
        * batch_size
        * kv_heads
        * sequence_length
        * head_dim
        * bytes_per_element
    )

def format_bytes(n):
    if n < 1024**2:
        return f"{n/1024:.2f} KB"
    if n < 1024**3:
        return f"{n/1024**2:.2f} MB"
    return f"{n/1024**3:.2f} GB"

for T in [1000, 5000, 10000, 25000]:
    b = theoretical_kv_cache_bytes(
        n_layers, n_kv_heads, T, head_dim,
        bytes_per_element=2
    )
    print(f"{T:>6,} tokens -> {format_bytes(b)}")

## 5. A first plot: theoretical KV-cache growth

The important prediction is:

\[
M_{KV} \propto T
\]

So doubling the context length should approximately double the KV-cache memory.

This is one of the clearest differences between **MHA** and **GQA/MQA**: reducing the number of KV heads directly reduces the memory required for the cache.

In [ ]:
context_lengths = np.array([500, 1000, 2000, 5000, 10000, 20000, 40000])

cache_mb = [
    theoretical_kv_cache_bytes(
        n_layers, n_kv_heads, T, head_dim,
        bytes_per_element=2
    ) / 1024**2
    for T in context_lengths
]

plt.figure(figsize=(8, 5))
plt.plot(context_lengths, cache_mb, marker="o")
plt.xlabel("Context length (tokens)")
plt.ylabel("Theoretical KV-cache size (MiB)")
plt.title(f"Theoretical KV-cache growth — {attention_type}")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Compare MHA, GQA, and MQA

For the same model depth, sequence length, and head dimension:

\[
M_{KV} \propto H_{KV}
\]

This means that, relative to MHA, the approximate cache memory ratio is:

\[
\frac{M_{\text{GQA}}}{M_{\text{MHA}}}
=
\frac{H_{KV,\text{GQA}}}{H_{Q}}
\]

and:

\[
\frac{M_{\text{MQA}}}{M_{\text{MHA}}}
=
\frac{1}{H_Q}
\]

This is why GQA and MQA can be attractive for long-context inference.

In [ ]:
# Compare hypothetical attention schemes using this model's query-head count.
schemes = {
    "MHA": n_q_heads,
    "GQA (example)": max(1, n_q_heads // 4),
    "MQA": 1,
}

T = 20000

rows = []
for name, kv_heads in schemes.items():
    b = theoretical_kv_cache_bytes(
        n_layers, kv_heads, T, head_dim,
        bytes_per_element=2
    )
    rows.append({
        "Architecture": name,
        "KV heads": kv_heads,
        "Cache MiB": b / 1024**2,
    })

pd.DataFrame(rows)

# Experiment 1 — Measuring prefill latency

## What is prefill?

The **prefill phase** processes the user's existing prompt/context.

For example:

```text
[1000 input tokens]
       |
       v
Transformer processes the prompt
       |
       v
KV cache is constructed
       |
       v
First generated token
```

Unlike decode, prefill can process many input tokens in parallel.

We will measure how long it takes to process prompts of different lengths.

### Prediction

As context length increases:

- prefill time should increase;
- the relationship may be approximately linear over some range;
- on real hardware, it may curve or show irregularities.

Do not expect a perfect line.

In [ ]:
def make_input_ids(length):
    # Use a repeated token sequence so we can control the exact token count.
    # We avoid relying on natural-language tokenization to produce exact lengths.
    token_id = tokenizer.eos_token_id
    if token_id is None:
        token_id = 0
    return torch.full(
        (1, length),
        token_id,
        dtype=torch.long,
        device=device,
    )

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def measure_prefill(length, repeats=3):
    input_ids = make_input_ids(length)

    # Warm-up
    _ = model(
        input_ids=input_ids,
        use_cache=True,
    )
    synchronize()

    times = []

    for _ in range(repeats):
        if device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        synchronize()
        start = time.perf_counter()

        outputs = model(
            input_ids=input_ids,
            use_cache=True,
        )

        synchronize()
        elapsed = time.perf_counter() - start
        times.append(elapsed)

    return np.median(times), outputs.past_key_values

# Start conservatively.
prefill_lengths = [128, 256, 512, 1024, 2048]

prefill_results = []

for T in prefill_lengths:
    elapsed, past = measure_prefill(T, repeats=3)
    prefill_results.append({
        "context_length": T,
        "prefill_seconds": elapsed,
        "prefill_ms": elapsed * 1000,
    })
    print(f"{T:>5} tokens: {elapsed*1000:.2f} ms")

prefill_df = pd.DataFrame(prefill_results)
prefill_df

## Plot 1 — Context length vs. prefill latency

Interpret the shape carefully.

The theoretical story is more nuanced than simply saying "attention is \(O(T^2)\)":

- The full training-style attention matrix is \(T \times T\).
- But optimized inference kernels exploit parallelism.
- Modern attention implementations can avoid explicitly materializing the full matrix.
- Hardware utilization changes as the problem size changes.

So your measured prefill curve is an **empirical systems measurement**, not a direct plot of asymptotic complexity.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    prefill_df["context_length"],
    prefill_df["prefill_ms"],
    marker="o"
)
plt.xlabel("Context length (tokens)")
plt.ylabel("Prefill latency (ms)")
plt.title("Measured Prefill Latency vs. Context Length")
plt.grid(True, alpha=0.3)
plt.show()

# Experiment 2 — Measuring decode latency as the KV cache grows

Now we simulate a different situation.

We first process a prompt of length \(T\), producing a KV cache.

Then we generate one additional token.

The model only needs to process the **new token**, but its query attends over the cached history.

Conceptually:

\[
Q_{T+1}
\rightarrow
[K_1,\ldots,K_T]
\]

and retrieves from:

\[
[V_1,\ldots,V_T]
\]

### Prediction

As \(T\) increases:

- the KV cache grows;
- the new query has more keys and values to attend over;
- decode latency may increase.

However, the measured curve may not be perfectly linear because the operation is often dominated by memory movement and GPU kernel behaviour rather than pure arithmetic.

In [ ]:
@torch.inference_mode()
def measure_decode_latency(context_length, repeats=10):
    input_ids = make_input_ids(context_length)

    # Build the KV cache.
    outputs = model(
        input_ids=input_ids,
        use_cache=True,
    )
    past_key_values = outputs.past_key_values

    next_token = torch.tensor(
        [[tokenizer.eos_token_id or 0]],
        dtype=torch.long,
        device=device,
    )

    # Warm-up decode
    _ = model(
        input_ids=next_token,
        past_key_values=past_key_values,
        use_cache=True,
    )
    synchronize()

    times = []

    for _ in range(repeats):
        synchronize()
        start = time.perf_counter()

        _ = model(
            input_ids=next_token,
            past_key_values=past_key_values,
            use_cache=True,
        )

        synchronize()
        times.append(time.perf_counter() - start)

    return np.median(times)

decode_lengths = [128, 256, 512, 1024, 2048]

decode_results = []

for T in decode_lengths:
    elapsed = measure_decode_latency(T, repeats=10)
    decode_results.append({
        "context_length": T,
        "decode_seconds": elapsed,
        "decode_ms": elapsed * 1000,
    })
    print(f"{T:>5} tokens: {elapsed*1000:.3f} ms/token")

decode_df = pd.DataFrame(decode_results)
decode_df

## Plot 2 — Context length vs. decode latency

This is the experiment most directly connected to the question:

> Why might a long conversation make each subsequent token slower?

The expected trend is upward, but don't be surprised if it is noisy.

On some hardware, the difference between 128 and 2048 tokens may be small. On other hardware, it may be much more visible.

The important lesson is that **KV caching changes the problem from recomputing the entire history to reading and attending over an increasingly large stored history**.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    decode_df["context_length"],
    decode_df["decode_ms"],
    marker="o"
)
plt.xlabel("Cached context length (tokens)")
plt.ylabel("Decode latency per token (ms)")
plt.title("Measured Decode Latency vs. KV-Cache Length")
plt.grid(True, alpha=0.3)
plt.show()

# Experiment 3 — Measuring the KV cache itself

The exact internal representation of `past_key_values` can vary between Transformers versions and model architectures.

We can nevertheless inspect the tensors returned by the model.

For each layer, we expect K and V tensors containing the cached sequence.

We will:

1. inspect the first layer;
2. count the total number of tensor elements;
3. calculate their actual byte footprint;
4. compare that with the theoretical estimate.

This is useful because it demonstrates the difference between:

- **theoretical KV-cache memory**;
- **actual tensor storage**;
- **total process/GPU memory**.

In [ ]:
@torch.inference_mode()
def inspect_kv_cache(context_length):
    input_ids = make_input_ids(context_length)

    outputs = model(
        input_ids=input_ids,
        use_cache=True,
    )

    past = outputs.past_key_values

    # Support both tuple-style caches and newer Cache objects where possible.
    if hasattr(past, "to_legacy_cache"):
        legacy = past.to_legacy_cache()
    else:
        legacy = past

    total_elements = 0
    total_bytes = 0
    tensor_shapes = []

    for layer_cache in legacy:
        for tensor in layer_cache[:2]:  # K and V
            total_elements += tensor.numel()
            total_bytes += tensor.numel() * tensor.element_size()

        if len(tensor_shapes) == 0:
            tensor_shapes = [tuple(x.shape) for x in layer_cache[:2]]

    return {
        "context_length": context_length,
        "elements": total_elements,
        "bytes": total_bytes,
        "MiB": total_bytes / 1024**2,
        "first_layer_KV_shapes": tensor_shapes,
    }

kv_results = []

for T in decode_lengths:
    result = inspect_kv_cache(T)
    kv_results.append(result)
    print(
        f"{T:>5} tokens: "
        f"{result['MiB']:.2f} MiB "
        f"| shapes: {result['first_layer_KV_shapes']}"
    )

kv_df = pd.DataFrame(kv_results)
kv_df

## Plot 3 — Context length vs. measured KV-cache memory

If the cache is storing one K and one V vector for each cached token, then:

\[
M_{KV} \propto T
\]

This is the cleanest scaling relationship in the practical.

If you see deviations, investigate:

- whether the model uses a cache implementation with extra metadata;
- whether the model uses grouped or shared KV heads;
- whether tensors are stored in FP16/BF16/FP32;
- whether the model uses a newer cache abstraction.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    kv_df["context_length"],
    kv_df["MiB"],
    marker="o"
)
plt.xlabel("Context length (tokens)")
plt.ylabel("KV-cache tensor memory (MiB)")
plt.title("Measured KV-Cache Memory vs. Context Length")
plt.grid(True, alpha=0.3)
plt.show()

# Experiment 4 — Put the three measurements together

We now compare:

1. **Prefill latency**
2. **Decode latency per token**
3. **KV-cache memory**

These quantities have different scaling behaviour.

A useful conceptual summary is:

| Quantity | Simplified scaling |
|---|---|
| KV-cache memory | \(O(T)\) |
| Decode attention for one new token | \(O(T)\) |
| Cumulative decode attention over \(T\) generated tokens | \(O(T^2)\) |
| Prefill | Depends strongly on attention implementation and hardware |

The last row is intentionally cautious.

Do not confuse a theoretical complexity bound with the wall-clock time of a highly optimized implementation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(
    prefill_df["context_length"],
    prefill_df["prefill_ms"],
    marker="o"
)
axes[0].set_title("Prefill")
axes[0].set_xlabel("Context length")
axes[0].set_ylabel("Milliseconds")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    decode_df["context_length"],
    decode_df["decode_ms"],
    marker="o"
)
axes[1].set_title("Decode")
axes[1].set_xlabel("Cached context length")
axes[1].set_ylabel("Milliseconds / token")
axes[1].grid(True, alpha=0.3)

axes[2].plot(
    kv_df["context_length"],
    kv_df["MiB"],
    marker="o"
)
axes[2].set_title("KV Cache")
axes[2].set_xlabel("Context length")
axes[2].set_ylabel("MiB")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 7. Why are the curves not perfectly linear?

This is an important discussion point.

Students often expect:

\[
T \uparrow
\quad\Rightarrow\quad
\text{latency} \uparrow \text{ perfectly linearly}
\]

But real systems are not simple mathematical functions.

## Reasons include

### 1. GPU kernel efficiency

Small operations may under-utilize the GPU. Larger operations may use hardware more efficiently.

### 2. Memory bandwidth

Decode can become memory-bandwidth bound. The model may spend more time moving K/V data than doing arithmetic.

### 3. Kernel fusion

Optimized implementations combine multiple operations into efficient GPU kernels.

### 4. Attention implementation

FlashAttention and other optimized attention mechanisms change the practical relationship between sequence length and runtime.

### 5. Hardware cache effects

Data may fit into different levels of CPU/GPU cache or memory hierarchy at different context lengths.

### 6. Allocator behaviour

Memory allocation and fragmentation can affect measurements.

### 7. OS and background load

Laptop experiments are inherently noisy.

### 8. Python overhead

For very small decode steps, the Python call itself can be a noticeable fraction of total runtime.

Therefore:

> **Asymptotic complexity tells us how costs scale in principle. Benchmarking tells us how a particular implementation behaves on particular hardware.**

# 8. Final exercise — Explain the divergence

Use your measurements to answer the following questions.

### Question 1

Why does KV-cache memory grow approximately linearly with context length?

### Question 2

Why does decode latency tend to increase with context length even though we are using a KV cache?

### Question 3

Why is the prefill phase fundamentally different from the decode phase?

### Question 4

Why might your measured decode-latency curve be noisy or non-linear?

### Question 5

Suppose you double the context from 4,000 to 8,000 tokens.

Would you expect:

- exactly double the KV-cache memory?
- exactly double the decode latency?
- exactly double the prefill latency?

Explain why these three answers may differ.

### Question 6 — Architecture

If a model has:

- 32 query heads,
- 32 layers,
- head dimension 128,
- FP16 KV tensors,

compare the theoretical KV-cache memory at 100,000 tokens for:

1. MHA: 32 KV heads
2. GQA: 8 KV heads
3. MQA: 1 KV head

Which architecture is most memory-efficient, and what trade-offs might it introduce?

### Question 7 — Application design

Imagine you are building a customer-support chatbot.

The conversation history can grow to 100,000 tokens.

Would you:

- send the entire history every time?
- summarize older messages?
- use retrieval?
- use a sliding window?
- combine these approaches?

Design a strategy that maximizes **useful context per unit of compute and memory**.

---

## Challenge

Repeat the experiments with:

1. a different model;
2. a different attention architecture, if available;
3. CPU vs. GPU;
4. FP32 vs. FP16/BF16 where supported.

Write one paragraph explaining which conclusions were stable across experiments and which depended on the hardware or model.

# 9. Key takeaways

### Takeaway 1

The KV cache prevents us from recomputing the keys and values of the entire history at every decoding step.

### Takeaway 2

The cache itself grows approximately linearly with sequence length:

\[
M_{KV} \propto T
\]

### Takeaway 3

Each new query still attends over the growing cached history.

Therefore, decode does not become constant-time just because we use a KV cache.

### Takeaway 4

Prefill and decode are different computational regimes.

### Takeaway 5

GQA and MQA reduce KV-cache memory by reducing the number of K/V heads.

### Takeaway 6

Theoretical complexity and measured wall-clock latency are not the same thing.

The practical engineering question is:

> **How do we give an LLM enough useful context without paying unnecessarily for irrelevant context?**